In [7]:
import pandas as pd
import joblib
import shap
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# -------------------------------
# LOAD ARTIFACTS (LOAD ONCE)
# -------------------------------
model = joblib.load("model.pkl")
tfidf = joblib.load("tfidf.pkl")
columns = joblib.load("columns.pkl")

explainer = shap.TreeExplainer(model)

# -------------------------------
# BUILD FEATURE ROW
# -------------------------------
def build_feature_row(startup_data, investor_data):

    df = pd.DataFrame([{
        "startup_industry": startup_data["industry"],
        "startup_stage": startup_data["stage"],
        "funding_required_lakhs": startup_data["funding_required"],

        "investor_pref_industry": investor_data["preferred_industry"],
        "investor_pref_stage": investor_data["preferred_stage"],
        "investor_ticket_min_lakhs": investor_data["ticket_min"],
        "investor_ticket_max_lakhs": investor_data["ticket_max"],

        "startup_desc": startup_data["description"],
        "investor_desc": investor_data["description"]
    }])

    # -------------------------------
    # IDEA SIMILARITY
    # -------------------------------
    s_vec = tfidf.transform(df["startup_desc"])
    i_vec = tfidf.transform(df["investor_desc"])
    similarity = cosine_similarity(s_vec, i_vec)[0][0]

    df["idea_similarity"] = float(similarity)

    # -------------------------------
    # FUNDING FIT
    # -------------------------------
    df["funding_fit"] = (
        (df["funding_required_lakhs"] >= df["investor_ticket_min_lakhs"]) &
        (df["funding_required_lakhs"] <= df["investor_ticket_max_lakhs"])
    ).astype(int)

    df = df.drop(["startup_desc", "investor_desc"], axis=1)

    # -------------------------------
    # ONE-HOT ENCODING + ALIGNMENT
    # -------------------------------
    df_encoded = pd.get_dummies(df, drop_first=True)
    df_aligned = df_encoded.reindex(columns=columns, fill_value=0)

    return df_aligned


# -------------------------------
# EXPLANATION ENGINE
# -------------------------------
def explain_match(startup_data, investor_data):

    X_input = build_feature_row(startup_data, investor_data)

    # -------------------------------
    # MODEL PREDICTION
    # -------------------------------
    prediction = model.predict(X_input)[0]
    prediction = float(np.clip(prediction, 0, 100))

    # -------------------------------
    # SHAP VALUES (ROBUST HANDLING)
    # -------------------------------
    shap_values = explainer.shap_values(X_input)

    # Handle different SHAP output formats
    if isinstance(shap_values, list):
        shap_vals = shap_values[0][0]
    else:
        shap_vals = shap_values[0]

    feature_contrib = dict(zip(X_input.columns, shap_vals))

    # -------------------------------
    # CONTRIBUTION BUCKETS
    # -------------------------------
    contributions = {
        "Industry Fit": 0.0,
        "Stage Fit": 0.0,
        "Funding Fit": 0.0,
        "Idea Similarity": 0.0
    }

    for feature, value in feature_contrib.items():

        f = feature.lower()

        if "industry" in f:
            contributions["Industry Fit"] += value

        elif "stage" in f:
            contributions["Stage Fit"] += value

        elif "funding_fit" in f:
            contributions["Funding Fit"] += value

        elif "idea_similarity" in f:
            contributions["Idea Similarity"] += value

    # -------------------------------
    # NORMALIZE FOR UI DISPLAY
    # -------------------------------
    total = sum(abs(v) for v in contributions.values()) + 1e-6

    for k in contributions:
        contributions[k] = round(100 * abs(contributions[k]) / total, 1)

    return {
        "match_score": round(prediction, 2),
        "contributions": contributions
    }